# Conversational App for Itinerary Planning

In [ ]:
import os
from dotenv import load_dotenv
#from langchain import HuggingFaceHub

from langchain import PromptTemplate, LLMChain

load_dotenv()

#os.environ["HUGGINGFACEHUB_API_TOKEN"]
#retrieving api key
key=os.environ['OPENAI_API_KEY']

In [ ]:
from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)
from langchain_core.messages import (
    AIMessage,
    HumanMessage,
    SystemMessage
)
from langchain.chains import LLMChain, ConversationChain
from langchain_openai import ChatOpenAI

chat = ChatOpenAI()

## Sample bot with no memory

In [ ]:
from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)
from langchain_core.messages import (
    AIMessage,
    HumanMessage,
    SystemMessage
)
from langchain.chains import LLMChain, ConversationChain
from langchain_openai import ChatOpenAI

chat = ChatOpenAI()
messages = [
    SystemMessage(content="You are a helpful assistant that help the user to plan an optimized itinerary."),
    HumanMessage(content="I'm going to Rome for 2 days, what can I visit?")
]

# Use invoke instead of the deprecated __call__ method
output = chat.invoke(messages)
print(output.content)

## Adding Memory

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from typing import List

# Initialize the chat model
chat = ChatOpenAI()

# Create a message history
message_history = ChatMessageHistory()

# Add system message
system_message = SystemMessage(content="You are a helpful assistant.")
message_history.add_message(system_message)

# Create a messages list
messages = message_history.messages + [
    HumanMessage(content="Hi there!")
]

# Run the conversation
response = chat.invoke(messages)

# Add the response to history
message_history.add_message(response)

In [ ]:
# Add a new human message for the follow-up question
new_question = HumanMessage(content="What is the most iconic place in Rome?")
message_history.add_message(new_question)

# Get all messages from the history
messages = message_history.messages

# Invoke the chat model with all messages
response = chat.invoke(messages)

# Add the response to history
message_history.add_message(response)

# Print the response
print(response.content)

In [ ]:
# Add a new human message for the follow-up question
new_question = HumanMessage(content="What kind of other events?")
message_history.add_message(new_question)

# Get all messages from the history
messages = message_history.messages

# Invoke the chat model with all messages
response = chat.invoke(messages)

# Add the response to history
message_history.add_message(response)

# Print the response
print(response.content)

In [ ]:
# Get all messages from the message history
messages = message_history.messages

In [ ]:
from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)
from langchain_community.chat_message_histories import ChatMessageHistory  # Try community package
from langchain_core.runnables import RunnableSequence
from langchain_openai import ChatOpenAI

# Initialize the chat model if not already done
chat = ChatOpenAI()

# Keep your prompt setup the same
prompt = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template(
        "You are a helpful assistant that help the user to plan an optimized itinerary."
    ),
    MessagesPlaceholder(variable_name="chat_history"),
    HumanMessagePromptTemplate.from_template("{question}")
])

# Set up message history instead of memory
chat_history = ChatMessageHistory()

# Create the chain using the pipe syntax
conversation = prompt | chat

# To use it with chat history, create an invoke function
async def invoke_conversation(question: str):
    # Get messages from chat history
    messages = chat_history.messages
    
    # Invoke conversation
    response = await conversation.ainvoke({
        "chat_history": messages,
        "question": question
    })
    
    # Add the new messages to history
    chat_history.add_user_message(question)
    chat_history.add_ai_message(response.content)
    
    return response

# Non-async version if needed
def invoke_conversation_sync(question: str):
    # Get messages from chat history
    messages = chat_history.messages
    
    # Invoke conversation
    response = conversation.invoke({
        "chat_history": messages,
        "question": question
    })
    
    # Add the new messages to history
    chat_history.add_user_message(question)
    chat_history.add_ai_message(response.content)
    
    return response

In [ ]:
while True:
    query = input('you: ')
    if query == 'q':
        break
    
    # Load memory contents
    memory_contents = memory.load_memory_variables({})
    
    # Use invoke() method instead of calling directly
    output = conversation.invoke({
        "chat_history": memory_contents["chat_history"],
        "question": query
    })
    
    # The output structure is different in the new format
    print('User: ', query)
    print('AI system: ', output.content)  # Use .content instead of ['text']
    
    # Save the conversation to memory
    memory.save_context({"question": query}, {"output": output.content})

## Adding non parametric knowledge

In [ ]:
from langchain.llms import OpenAI
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.document_loaders import PyPDFLoader

import os

from dotenv import load_dotenv

load_dotenv()

os.environ["OPENAI_API_KEY"]

text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=1500,
            chunk_overlap=200
        )

raw_documents = PyPDFLoader('italy_travel.pdf').load()
documents = text_splitter.split_documents(raw_documents)
db = FAISS.from_documents(documents, OpenAIEmbeddings())

In [ ]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

# Create a message history to store the conversation
message_history = ChatMessageHistory()

# Initialize the LLM
llm = ChatOpenAI()

# Create prompt template
prompt = ChatPromptTemplate.from_template("""
Answer the following question based on the provided context:

Context: {context}

Question: {input}
""")

# Create the chain
document_chain = create_stuff_documents_chain(llm, prompt)
retrieval_chain = create_retrieval_chain(db.as_retriever(), document_chain)

# Use invoke instead of run
response = retrieval_chain.invoke({"input": "Give me some review about the Pantheon"})

# Add the question and answer to the message history
message_history.add_user_message("Give me some review about the Pantheon")
message_history.add_ai_message(response["answer"])

# Print the answer
print(response["answer"])

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain.chains import ConversationalRetrievalChain
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

# Custom template for standalone question generation
custom_template = """Given the following conversation and a follow up question, rephrase the follow up question to be a standalone question. 
If you cannot find the answer in the document provided, ignore the document and answer anyway.
Chat History:
{chat_history}
Follow Up Input: {question}
Standalone question:"""

CUSTOM_QUESTION_PROMPT = PromptTemplate.from_template(custom_template)

# Create the chain without memory
qa_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=db.as_retriever(),
    condense_question_prompt=CUSTOM_QUESTION_PROMPT,
    verbose=True
)

# Create a message history store
message_histories = {}

# Create a unique session ID
session_id = "user_session_1"

# Wrap the chain with message history
qa_chain_with_history = RunnableWithMessageHistory(
    qa_chain,
    lambda session_id: message_histories.setdefault(session_id, ChatMessageHistory()),
    input_messages_key="question",
    history_messages_key="chat_history"
)

# Use it with the session ID
response = qa_chain_with_history.invoke(
    {"question": "What can I visit in India?"}, 
    config={"configurable": {"session_id": session_id}}
)

print(response)

In [ ]:
from langchain.agents.agent_toolkits import create_retriever_tool

tool = create_retriever_tool(
    db.as_retriever(), 
    "italy_travel",
    "Searches and returns documents regarding Italy."
)
tools = [tool]

memory = ConversationBufferMemory(
            memory_key='chat_history',
            return_messages=True
        )

from langchain.agents.agent_toolkits import create_conversational_retrieval_agent

from langchain_openai import ChatOpenAI
llm = ChatOpenAI(temperature = 0)

agent_executor = create_conversational_retrieval_agent(llm, tools, memory_key='chat_history', verbose=True)

In [ ]:
agent_executor.invoke({"input": "hi, i'm Vale"})

In [ ]:
agent_executor.invoke({"input": "Tell me something about Pantheon"})

In [ ]:
# Use invoke instead of the call operator
output = agent_executor.invoke({"input": "what can I visit in India in 3 days?"})
output['output']

## Adding external tools

In [ ]:
from langchain import SerpAPIWrapper
from langchain.agents import AgentType, initialize_agent
from langchain.llms import OpenAI
from langchain.tools import BaseTool, StructuredTool, Tool, tool

import os
from dotenv import load_dotenv

load_dotenv()

key = os.environ["SERPAPI_API_KEY"]

search = SerpAPIWrapper()

In [ ]:
tools = [
    Tool.from_function(
        func=search.run,
        name="Search",
        description="useful for when you need to answer questions about current events"
    ),
    create_retriever_tool(
        db.as_retriever(), 
        "italy_travel",
        "Searches and returns documents regarding Italy."
    )
    ]

agent_executor = create_conversational_retrieval_agent(llm, tools, memory_key='chat_history', verbose=True)

In [ ]:
memory

In [ ]:
agent_executor.invoke({"input": "what can I visit in India in 3 days?"})

In [ ]:
agent_executor({"input": "what is the current wheather in Delhi ?"})

In [ ]:
agent_executor({"input": "I'm travelling to Italy, can you give me some suggestions of the main attractions?"})